# Tabular foundation model as the cell model: TabPFN v2 on IMC (`future_plan.md` §18.8)

**Question.** The per-cell table model (LightGBM) is the best virtual-cell model we have. Does a tabular foundation model, which predicts from a context of labelled cells with no training and returns a predictive distribution, do better, especially under cohort shift and in few-shot transfer?

**Model.** TabPFN v2 regressor (`tabpfn==9.2.0`, `ModelVersion.V2`; open weights, no licence gate; context limit 10,000 rows), 4 estimators, one T4. Quantiles at the 9 `prob-impute` levels → 50 draws per cell by the same inverse-CDF interpolation.

**Comparators** (same inputs: 14 other shared channels + log area; same scaling as `prob-impute`):
- *LightGBM (full)*: point + 9-quantile models on up to 300k / 100k cells per cohort (= `prob-impute`).
- *LightGBM (same context)*: the same models fitted on exactly TabPFN's 10k context cells, so the comparison at equal data is fair.

**A. Zero-shot grid** (replicate-0 split): train on {Jackson, Danenberg, Both} × test on {Jackson, Danenberg}.
**B. Few-shot transfer** (J→D, D→J): k ∈ {5, 20} patients from the target cohort's *training* split join the context (TabPFN: 5k source + up to 5k target cells; LightGBM: full source + all k-patient cells, weighted to equal total weight). A cohort flag is added as a feature. Everything is in the source cohort's scaling, so the target offset can only be learned from the k patients.

**Scoring:** a fixed random subsample of 40,000 test cells per cohort, the same for every model. CRPS, coverage (50/80/95), PIT KS, W2 on one draw, R4 of the median, point RMSE; 10 markers (ERα excluded from means); 1000× patient bootstrap, paired CRPS differences vs LightGBM (full).

**Decision (fixed before the run, §18.8):** TabPFN is a claim only if it beats LightGBM (full) on transfer coverage (mean |cov₈₀ − 0.80| lower, paired CI) or on few-shot CRPS (paired CI < 0 for ≥ 6/10 markers). Within-cohort R4 alone does not count.

*Cell typing (§18.8 task 2) is deferred to a second run: TabPFN v2 caps the classifier at 10 classes, so only the 6 coarse classes fit, and they need the `cell-type-graph` label mapping.*

In [ ]:
import os
os.environ["SCIPY_ARRAY_API"] = "1"      # TabPFN enables sklearn array-API dispatch, which requires this before scipy loads
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn==9.2.0"], check=True)
import os, glob, json, time, sys, subprocess, warnings, traceback
import numpy as np, pandas as pd
import pyarrow.parquet as pq
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb
from sklearn.mixture import GaussianMixture
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013
REP = 0
N_BOOT = 100 if SMOKE else 1000
MIN_CELLS = 200
COF = 5.0
N_FIT_FINAL = 30_000 if SMOKE else 300_000    # cells per cohort for the point model (as train-test-grid)
N_FIT_Q = 20_000 if SMOKE else 100_000        # cells per cohort for each quantile model
N_DRAW = 50
QL = np.array([0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975])
LGB = dict(n_estimators=40 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
LGBQ = dict(LGB, n_estimators=40 if SMOKE else 150)
NUC = ["histone_h3", "dna1", "dna2"]
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)
assert len(MARKERS) == 11, MARKERS
EXCL = "estrogen_receptor_alpha"
SOURCES = ["Jackson", "Danenberg", "Both"]
COHORTS = ["Jackson", "Danenberg"]
OTHER = {"Jackson": "Danenberg", "Danenberg": "Jackson"}
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

FAILED = []
def guard(name):
    def deco(fn):
        try:
            return fn()
        except Exception:
            print(f"!!! block {name} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(name)
            return None
    return deco
import torch
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_CTX = 10_000
N_TEST = 40_000
KSHOT = [5, 20]
TPF_EST = 4
CHUNK = 4_000
TIME_LIMIT_MIN = 600
print("device", DEVICE, torch.__version__)

## Cell tables and Danenberg per-cell image means (as `prob-impute`)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = dict(cells=p)
P["danenberg"]["split"] = near("danenberg_patient_split.csv", P["danenberg"]["cells"])

def load_table(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    return dict(cells=cells, img=img_code, img_names=np.asarray(img_names), starts=starts,
                patient=cells[pid].astype(str).values,
                log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32))

DD, DJ = load_table("danenberg"), load_table("jackson")
split = pd.read_csv(P["danenberg"]["split"]).set_index("metabric_id").split
DD["split"] = pd.Series(DD["patient"]).map(split).fillna("none").values
tick(f"tables: Danenberg {len(DD['cells']):,} cells, Jackson {len(DJ['cells']):,} cells")

if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs==2024.9.22"]).returncode:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs"], check=True)
import tifffile
tifs = g("MB_*.tiff")
D_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" not in p.replace("\\", "/")}
D_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" in p.replace("\\", "/")}
OBJ = DD["cells"].object_number.values.astype(np.int64)
ST = DD["starts"]

def raw_means(img_path, mask_path, obj):
    m = tifffile.imread(mask_path).astype(np.int64)
    lut = np.full(max(int(m.max()), int(obj.max())) + 1, -1, np.int64)
    lut[obj] = np.arange(len(obj))
    loc = lut[m.ravel()]
    v = np.flatnonzero(loc >= 0)
    lv = loc[v]
    cnt = np.bincount(lv, minlength=len(obj))
    im = tifffile.imread(img_path)
    if im.ndim == 3 and im.shape[0] == len(D_CHAN) and im.shape[-1] != len(D_CHAN):
        im = np.moveaxis(im, 0, -1)
    im = im.reshape(-1, im.shape[-1]).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.stack([np.where(cnt > 0, np.bincount(lv, im[v, c], len(obj)) / np.maximum(cnt, 1), np.nan)
                         for c in range(im.shape[1])], 1).astype(np.float32)

RAW = np.vstack(Parallel(n_jobs=os.cpu_count())(delayed(raw_means)(D_IMG[s], D_MASK[s], OBJ[a:b])
                                                for s, a, b in zip(DD["img_names"], ST[:-1], ST[1:])))
assert RAW.shape == (len(DD["cells"]), len(D_CHAN))
dix = {m: i for i, m in enumerate(D_CHAN)}
IN_SET = NUC + MARKERS
X = {"Danenberg": np.c_[np.arcsinh(RAW[:, [dix[c] for c in IN_SET]] / COF), DD["log_area"]].astype(np.float64),
     "Jackson": np.c_[np.arcsinh(DJ["cells"][IN_SET].values.astype(np.float64) / COF), DJ["log_area"]].astype(np.float64)}
del RAW
tick("per-cell image means")

## Splits and per-cohort scaling statistics (as `prob-impute`)

In [ ]:
jp, jcode = np.unique(DJ["patient"], return_inverse=True)
u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
D_TRVA_PAT = np.unique(DD["patient"][DD["split"] != "test"])
uv = np.random.default_rng(SEED + 21 + REP).random(len(D_TRVA_PAT))
dval = set(D_TRVA_PAT[uv < 0.125])
D_SPLIT = np.where(DD["split"] == "test", "test", np.where(pd.Series(DD["patient"]).isin(dval).values, "val", "train"))
SPLIT = {"Jackson": J_SPLIT, "Danenberg": D_SPLIT}
IDX = {c: {s: np.flatnonzero(SPLIT[c] == s) for s in ("train", "val", "test")} for c in COHORTS}
PAT = {"Jackson": DJ["patient"], "Danenberg": DD["patient"]}
IMG = {"Jackson": DJ["img"], "Danenberg": DD["img"]}
for c in COHORTS:
    print(c, {s: (len(v), len(set(PAT[c][v]))) for s, v in IDX[c].items()}, "(cells, patients)")

# harmonisation statistics, exactly as harm() in train-test-grid (all cells of the cohort)
STAT = {}
for c in COHORTS:
    med = np.nanmedian(X[c], 0)
    q05, q95 = np.nanpercentile(X[c], [5, 95], 0)
    STAT[c] = (med, np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X[c], 0) + 1e-6))

def H(c_data, c_stat):
    med, sc = STAT[c_stat]
    return ((X[c_data] - med) / sc).astype(np.float32)

HH = {(cd, cs): H(cd, cs) for cd in COHORTS for cs in COHORTS}
tick("splits and harmonisation")

## Scoring helpers (as `prob-impute`)

In [ ]:
def crps_draws(S, y):
    # S: (n, k) draws, y: (n,). CRPS = E|X - y| - 0.5 E|X - X'|, the unbiased ("fair") estimator over draws
    k = S.shape[1]
    Ss = np.sort(S, 1)
    t1 = np.abs(Ss - y[:, None]).mean(1)
    w = (2 * np.arange(1, k + 1) - k - 1) / (k * (k - 1))
    t2 = (Ss * w).sum(1)          # = 0.5 E|X - X'|
    return t1 - t2

def boot_mean(v, pc, n_pat, W):
    s, c = np.bincount(pc, v, n_pat), np.bincount(pc, minlength=n_pat).astype(np.float64)
    return float(s.sum() / c.sum()), (W @ s) / (W @ c)

def r4(a, b, img):
    ok = np.isfinite(a) & np.isfinite(b)
    a, b, im = a[ok], b[ok], img[ok]
    k = int(im.max()) + 1
    cnt = np.maximum(np.bincount(im, minlength=k), 1)
    a = a - (np.bincount(im, a, k) / cnt)[im]
    b = b - (np.bincount(im, b, k) / cnt)[im]
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))

def interp_draws(Q, rng, n):
    # inverse-CDF draws from per-cell quantiles Q (cells × levels), linear between levels; tails held flat
    # at the outer levels beyond 0.025 / 0.975 (i.e. the outer 5% is clamped)
    uq = rng.random((Q.shape[0], n))
    out = np.empty_like(uq)
    for i in range(n):
        u_ = np.clip(uq[:, i], QL[0], QL[-1])
        j = np.clip(np.searchsorted(QL, u_) - 1, 0, len(QL) - 2)
        t = (u_ - QL[j]) / (QL[j + 1] - QL[j])
        r = np.arange(Q.shape[0])
        out[:, i] = Q[r, j] * (1 - t) + Q[r, j + 1] * t
    return out

TEST = {c: IDX[c]["test"] for c in COHORTS}
BOOTW = {}
for c in COHORTS:
    tp, tcode = np.unique(PAT[c][TEST[c]], return_inverse=True)
    W = np.random.default_rng(SEED + (12 if c == "Jackson" else 13)).multinomial(
        len(tp), np.full(len(tp), 1 / len(tp)), size=N_BOOT).astype(np.float64)
    BOOTW[c] = (tcode, len(tp), W)

## Fixed test subsample and bootstrap weights

In [ ]:
for c in COHORTS:
    full = IDX[c]["test"]
    TEST[c] = np.sort(np.random.default_rng(SEED + 41).choice(full, min(N_TEST, len(full)), replace=False))
    tp, tcode = np.unique(PAT[c][TEST[c]], return_inverse=True)
    W = np.random.default_rng(SEED + (12 if c == "Jackson" else 13)).multinomial(
        len(tp), np.full(len(tp), 1 / len(tp)), size=N_BOOT).astype(np.float64)
    BOOTW[c] = (tcode, len(tp), W)
    print(c, "test cells scored", len(TEST[c]), "patients", len(tp))

def tabpfn_fit(Xc, yc, seed):
    m = TabPFNRegressor.create_default_for_version(ModelVersion.V2, device=DEVICE, n_estimators=TPF_EST,
                                                   random_state=seed, ignore_pretraining_limits=True)
    return m.fit(Xc, yc)

def tabpfn_quantiles(m, Xq):
    out = []
    for a in range(0, len(Xq), CHUNK):
        q = m.predict(Xq[a:a + CHUNK], output_type="quantiles", quantiles=[float(v) for v in QL])
        out.append(np.stack(q, 1))
    return np.sort(np.vstack(out), 1)

# smoke check of the TabPFN call on synthetic data
_r = np.random.default_rng(0); _X = _r.normal(size=(500, 5)); _y = _X[:, 0] + 0.1 * _r.normal(size=500)
_Q = tabpfn_quantiles(tabpfn_fit(_X, _y, 0), _X[:50])
assert _Q.shape == (50, len(QL)) and np.corrcoef(_Q[:, 4], _y[:50])[0, 1] > 0.9
tick("TabPFN v2 loads and predicts")

## Scoring

In [ ]:
ROWS, LOG, BOOTS = [], [], {}

def score(setting, src_, test, m, name, draws=None, point=None):
    k = IN_SET.index(m)
    y = X[test][TEST[test], k]
    ok = np.isfinite(y)
    tcode, n_pat, W = BOOTW[test]
    pc, y = tcode[ok], y[ok]
    img = IMG[test][TEST[test]][ok]
    rec = dict(setting=setting, train=src_, test=test, marker=m, model=name)
    key = (setting, src_, test, m, name)
    if draws is not None:
        S = draws[ok]
        crps = crps_draws(S, y)
        for lev in (0.5, 0.8, 0.95):
            lo, hi = np.quantile(S, (1 - lev) / 2, 1), np.quantile(S, 1 - (1 - lev) / 2, 1)
            pt, bv = boot_mean(((y >= lo) & (y <= hi)).astype(np.float64), pc, n_pat, W)
            rec[f"cov{int(lev * 100)}"] = pt
            BOOTS[key + (f"cov{int(lev * 100)}",)] = (pt, bv)
            if lev == 0.8:
                rec["width80"] = float(np.mean(hi - lo))
        Ss = np.sort(S, 1)
        pit = ((Ss < y[:, None]).sum(1) + np.random.default_rng(SEED + 7).random(len(y)) * ((Ss == y[:, None]).sum(1) + 1)) / (S.shape[1] + 1)
        ps = np.sort(pit)
        rec["pit_ks"] = float(np.max(np.abs(ps - (np.arange(1, len(ps) + 1) - 0.5) / len(ps))))
        one = S[np.arange(len(y)), np.random.default_rng(SEED + 9).integers(0, S.shape[1], len(y))]
        rec["W2_draw"] = float(np.sqrt(np.mean((np.sort(one) - np.sort(y)) ** 2)))
        rec["R4"] = r4(y, np.median(S, 1), img)
    else:
        p = point[ok]
        crps = np.abs(p - y)
        rec["rmse"] = float(np.sqrt(np.mean((p - y) ** 2)))
        rec["R4"] = r4(y, p, img) if np.std(p) > 0 else np.nan
    pt, bv = boot_mean(crps, pc, n_pat, W)
    rec["crps"] = pt
    BOOTS[key + ("crps",)] = (pt, bv)
    ROWS.append(rec)

def lgb_models(Xtr, ytr, Xq, yq, wtr=None, wq=None):
    point = lgb.LGBMRegressor(random_state=SEED + REP, **LGB).fit(Xtr, ytr, sample_weight=wtr)
    qms = [lgb.LGBMRegressor(objective="quantile", alpha=float(a), random_state=SEED + REP, **LGBQ).fit(Xq, yq, sample_weight=wq)
           for a in QL]
    return point, qms

def score_all(setting, src_, test, m, Hin, back, tpf, lgb_full, lgb_ctx):
    rng = np.random.default_rng(SEED + 31)
    Q = tabpfn_quantiles(tpf, Hin)
    score(setting, src_, test, m, "TabPFN v2", draws=back(interp_draws(Q, rng, N_DRAW)))
    score(setting, src_, test, m, "TabPFN v2 (median)", point=back(Q[:, 4]))
    for tag, (pm, qms) in (("LightGBM (full)", lgb_full), ("LightGBM (same context)", lgb_ctx)):
        Ql = np.sort(np.stack([q.predict(Hin) for q in qms], 1), 1)
        score(setting, src_, test, m, f"Quantile {tag}", draws=back(interp_draws(Ql, rng, N_DRAW)))
        score(setting, src_, test, m, f"Point {tag}", point=back(pm.predict(Hin)))

def mapping(src_, test):
    if src_ == "Both" or src_ == test:
        return test, test
    return src_, src_

def over_time():
    return (time.time() - T0) / 60 > TIME_LIMIT_MIN

## A. Zero-shot grid: 3 training sets × 11 markers × 2 test cohorts

In [ ]:
@guard("zero-shot grid")
def _():
    for src_ in SOURCES:
        for m in MARKERS:
            if over_time():
                print("time limit: skipping", src_, m); continue
            k = IN_SET.index(m)
            cols = [i for i in range(len(IN_SET) + 1) if i != k]
            parts = COHORTS if src_ == "Both" else [src_]
            rs = np.random.default_rng(SEED + REP)
            Xtr, ytr, Xq, yq, Xc, yc = [], [], [], [], [], []
            for c in parts:
                Hc = HH[(c, c)]
                tr = IDX[c]["train"][np.isfinite(Hc[IDX[c]["train"], k])]
                f1 = np.sort(rs.choice(tr, min(N_FIT_FINAL, len(tr)), replace=False))
                fq = np.sort(rs.choice(tr, min(N_FIT_Q, len(tr)), replace=False))
                fc = np.sort(rs.choice(tr, min(N_CTX // len(parts), len(tr)), replace=False))
                Xtr.append(Hc[f1][:, cols]); ytr.append(Hc[f1, k])
                Xq.append(Hc[fq][:, cols]); yq.append(Hc[fq, k])
                Xc.append(Hc[fc][:, cols]); yc.append(Hc[fc, k])
            Xtr, ytr, Xq, yq, Xc, yc = (np.vstack(Xtr), np.concatenate(ytr), np.vstack(Xq), np.concatenate(yq),
                                        np.vstack(Xc), np.concatenate(yc))
            t0 = time.time()
            lgb_full = lgb_models(Xtr, ytr, Xq, yq)
            lgb_ctx = lgb_models(Xc, yc, Xc, yc)
            tpf = tabpfn_fit(Xc, yc, SEED + REP)
            for test in COHORTS:
                cin, cout = mapping(src_, test)
                med_o, sc_o = STAT[cout][0][k], STAT[cout][1][k]
                score_all("zero-shot", src_, test, m, HH[(test, cin)][TEST[test]][:, cols],
                          lambda v: v * sc_o + med_o, tpf, lgb_full, lgb_ctx)
            LOG.append(dict(setting="zero-shot", train=src_, marker=m, seconds=time.time() - t0))
            tick(f"zero-shot {src_}: {m} ({time.time() - t0:.0f} s)")
        pd.DataFrame(ROWS).to_csv(f"{OUT}/scores_per_marker.csv", index=False)

## B. Few-shot transfer: k target patients join the context

In [ ]:
@guard("few-shot")
def _():
    for s in COHORTS:
        t = OTHER[s]
        trp = np.unique(PAT[t][IDX[t]["train"]])
        for kk in KSHOT:
            pats = set(np.random.default_rng(SEED + 500 + kk).choice(trp, kk, replace=False))
            tgt_all = IDX[t]["train"][pd.Series(PAT[t][IDX[t]["train"]]).isin(pats).values]
            for m in MARKERS:
                if over_time():
                    print("time limit: skipping", s, kk, m); continue
                k = IN_SET.index(m)
                cols = [i for i in range(len(IN_SET) + 1) if i != k]
                Hs, Ht = HH[(s, s)], HH[(t, s)]          # everything in the source cohort's scaling
                rs = np.random.default_rng(SEED + REP)
                tr = IDX[s]["train"][np.isfinite(Hs[IDX[s]["train"], k])]
                tg = tgt_all[np.isfinite(Ht[tgt_all, k])]
                f1 = np.sort(rs.choice(tr, min(N_FIT_FINAL, len(tr)), replace=False))
                fq = np.sort(rs.choice(tr, min(N_FIT_Q, len(tr)), replace=False))
                n_t = min(N_CTX // 2, len(tg))
                fct = np.sort(rs.choice(tg, n_t, replace=False))
                fcs = np.sort(rs.choice(tr, N_CTX - n_t, replace=False))
                flag = lambda A, v: np.c_[A, np.full(len(A), v, np.float32)]
                Xs1, Xsq, Xsc = flag(Hs[f1][:, cols], 0), flag(Hs[fq][:, cols], 0), flag(Hs[fcs][:, cols], 0)
                Xt1, Xtc = flag(Ht[tg][:, cols], 1), flag(Ht[fct][:, cols], 1)
                ys1, ysq, ysc, yt1, ytc = Hs[f1, k], Hs[fq, k], Hs[fcs, k], Ht[tg, k], Ht[fct, k]
                t0 = time.time()
                w1 = np.r_[np.ones(len(ys1)), np.full(len(yt1), len(ys1) / len(yt1))]
                wq = np.r_[np.ones(len(ysq)), np.full(len(yt1), len(ysq) / len(yt1))]
                lgb_full = lgb_models(np.vstack([Xs1, Xt1]), np.r_[ys1, yt1], np.vstack([Xsq, Xt1]), np.r_[ysq, yt1], w1, wq)
                Xc, yc = np.vstack([Xsc, Xtc]), np.r_[ysc, ytc]
                lgb_ctx = lgb_models(Xc, yc, Xc, yc)
                tpf = tabpfn_fit(Xc, yc, SEED + REP)
                med_o, sc_o = STAT[s][0][k], STAT[s][1][k]
                score_all(f"few-shot k={kk}", s, t, m, flag(HH[(t, s)][TEST[t]][:, cols], 1),
                          lambda v: v * sc_o + med_o, tpf, lgb_full, lgb_ctx)
                LOG.append(dict(setting=f"few-shot k={kk}", train=s, marker=m, seconds=time.time() - t0,
                                n_target_cells=len(tg)))
                tick(f"few-shot {s}->{t} k={kk}: {m} ({len(tg)} target cells, {time.time() - t0:.0f} s)")
            pd.DataFrame(ROWS).to_csv(f"{OUT}/scores_per_marker.csv", index=False)
pd.DataFrame(ROWS).to_csv(f"{OUT}/scores_per_marker.csv", index=False)
pd.DataFrame(LOG).to_csv(f"{OUT}/training_log.csv", index=False)

## Summaries, paired differences vs LightGBM (full), and the decision

In [ ]:
R = pd.DataFrame(ROWS)
R10 = R[R.marker != EXCL].copy()
R10["kind"] = np.where(R10.setting != "zero-shot", "few-shot",
                       np.where(R10.train == R10.test, "own", np.where(R10.train == "Both", "both", "transfer")))
mets = [c for c in ["crps", "cov50", "cov80", "cov95", "width80", "pit_ks", "W2_draw", "R4", "rmse"] if c in R10]
SUM = R10.groupby(["setting", "kind", "train", "test", "model"])[mets].mean().reset_index()
SUM["gap80"] = R10.assign(g=(R10.cov80 - 0.8).abs()).groupby(["setting", "kind", "train", "test", "model"]).g.mean().values
SUM.to_csv(f"{OUT}/summary_mean10.csv", index=False)
with pd.option_context("display.width", 250, "display.max_columns", 20):
    print(SUM.round(3).to_string(index=False))

ms10 = [m for m in MARKERS if m != EXCL]
combos = R10[["setting", "train", "test"]].drop_duplicates().itertuples(index=False)
AGG, DIFF = [], []
for st, s_, t_ in combos:
    for name in R10.model.unique():
        for met in ["crps", "cov80"]:
            keys = [(st, s_, t_, m, name, met) for m in ms10]
            if not all(k_ in BOOTS for k_ in keys):
                continue
            AGG.append(dict(setting=st, train=s_, test=t_, model=name, metric=met,
                            value=np.mean([BOOTS[k_][0] for k_ in keys]),
                            lo=np.percentile(np.mean([BOOTS[k_][1] for k_ in keys], 0), 2.5),
                            hi=np.percentile(np.mean([BOOTS[k_][1] for k_ in keys], 0), 97.5)))
            if met == "cov80":
                gv = np.mean([np.abs(BOOTS[k_][1] - 0.8) for k_ in keys], 0)
                AGG.append(dict(setting=st, train=s_, test=t_, model=name, metric="gap80",
                                value=np.mean([abs(BOOTS[k_][0] - 0.8) for k_ in keys]),
                                lo=np.percentile(gv, 2.5), hi=np.percentile(gv, 97.5)))
    for name, ref in (("TabPFN v2", "Quantile LightGBM (full)"), ("TabPFN v2", "Quantile LightGBM (same context)"),
                      ("TabPFN v2 (median)", "Point LightGBM (full)")):
        for m in ms10:
            a, b = (BOOTS.get((st, s_, t_, m, n_, "crps")) for n_ in (name, ref))
            if a is None or b is None:
                continue
            lo, hi = np.percentile(a[1] - b[1], [2.5, 97.5])
            DIFF.append(dict(setting=st, train=s_, test=t_, model=name, vs=ref, marker=m, diff=a[0] - b[0], lo=lo, hi=hi,
                             call="better" if hi < 0 else "worse" if lo > 0 else "n.s."))
        # gap80 difference TabPFN vs quantile LightGBM (full), paired
        if name == "TabPFN v2":
            ka = [(st, s_, t_, m, name, "cov80") for m in ms10]
            kb = [(st, s_, t_, m, ref, "cov80") for m in ms10]
            if all(k_ in BOOTS for k_ in ka + kb) and ref == "Quantile LightGBM (full)":
                d = np.mean([np.abs(BOOTS[x][1] - 0.8) for x in ka], 0) - np.mean([np.abs(BOOTS[x][1] - 0.8) for x in kb], 0)
                AGG.append(dict(setting=st, train=s_, test=t_, model="TabPFN − LightGBM (full)", metric="gap80_diff",
                                value=float(np.mean([abs(BOOTS[x][0] - 0.8) for x in ka]) - np.mean([abs(BOOTS[x][0] - 0.8) for x in kb])),
                                lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5)))
AGG, DIFF = pd.DataFrame(AGG), pd.DataFrame(DIFF)
AGG.to_csv(f"{OUT}/summary_mean10_ci.csv", index=False)
DIFF.to_csv(f"{OUT}/crps_differences.csv", index=False)
print(AGG.pivot_table(index=["setting", "train", "test", "model"], columns="metric", values="value").round(3).to_string())
print(DIFF.groupby(["setting", "train", "test", "model", "vs"]).call.value_counts().unstack(fill_value=0).to_string())

# decision (fixed before the run)
tr_ = AGG[(AGG.metric == "gap80_diff") & (AGG.setting == "zero-shot") & (AGG.train != "Both") & (AGG.train != AGG.test)]
cov_win = {r.test: bool(r.hi < 0) for r in tr_.itertuples()}
fs = DIFF[(DIFF.setting != "zero-shot") & (DIFF.model == "TabPFN v2") & (DIFF.vs == "Quantile LightGBM (full)")]
fs_win = {f"{r[0]} {r[1]}->{r[2]}": int(v) for r, v in fs.assign(b=fs.call == "better").groupby(["setting", "train", "test"]).b.sum().items()}
decision = dict(transfer_coverage_better=cov_win, fewshot_crps_markers_better=fs_win,
                claim=bool(any(cov_win.values()) or any(v >= 6 for v in fs_win.values())),
                rule="TabPFN lowers transfer |cov80-0.8| vs quantile LightGBM (full) (paired CI<0), or few-shot CRPS better for >=6/10 markers")
print(json.dumps(decision, indent=1))
json.dump(decision, open(f"{OUT}/decision.json", "w"), indent=1)

## Figure: CRPS and 80% coverage, TabPFN vs LightGBM

In [ ]:
@guard("figure")
def _():
    names = ["Quantile LightGBM (full)", "Quantile LightGBM (same context)", "TabPFN v2"]
    colr = {"Quantile LightGBM (full)": "#2a78d6", "Quantile LightGBM (same context)": "#9ec5ee", "TabPFN v2": "#c2410c"}
    panels = [(t, [("zero-shot", t), ("zero-shot", "Both"), ("zero-shot", OTHER[t])] +
               [(f"few-shot k={kk}", OTHER[t]) for kk in KSHOT]) for t in COHORTS]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for i, (t, combos_) in enumerate(panels):
        labs = [f"own ({t})", "both", f"transfer\n({OTHER[t]}-trained)"] + [f"transfer\n+{kk} patients" for kk in KSHOT]
        for j, met in enumerate(["crps", "cov80"]):
            ax = axes[j, i]
            for qn, nm in enumerate(names):
                xs, vs, los, his = [], [], [], []
                for p_, (st, s_) in enumerate(combos_):
                    r = AGG[(AGG.setting == st) & (AGG.train == s_) & (AGG.test == t) & (AGG.model == nm) & (AGG.metric == met)]
                    if len(r):
                        xs.append(p_ + (qn - 1) * 0.25); vs.append(r.value.iloc[0]); los.append(r.lo.iloc[0]); his.append(r.hi.iloc[0])
                if xs:
                    vs, los, his = map(np.array, (vs, los, his))
                    ax.bar(xs, vs, width=0.22, color=colr[nm], label=nm)
                    ax.errorbar(xs, vs, yerr=[vs - los, his - vs], fmt="none", ecolor="#0b0b0b", elinewidth=0.8, capsize=2)
            if met == "cov80":
                ax.axhline(0.8, color="#0b0b0b", lw=1, ls=(0, (4, 3))); ax.set_ylim(0, 1)
            ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs, fontsize=8)
            ax.set_ylabel(("CRPS, arcsinh units (lower is better)" if met == "crps" else "80% interval coverage") + f"\ntest: {t}")
            ax.spines[["top", "right"]].set_visible(False)
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=3, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(f"{OUT}/fig_tabpfn.png", dpi=200)

json.dump({"failed_blocks": FAILED, "minutes": (time.time() - T0) / 60}, open(f"{OUT}/run_info.json", "w"))
print("failed blocks:", FAILED if FAILED else "none")